In [1]:
import subprocess
import sys
import importlib.metadata as metadata

packages = [
    "qiskit==2.2.3", "numpy==2.2.6", "scipy==1.15.3",
    "scikit-learn==1.6.1", "pandas==2.2.3", "matplotlib==3.10.1",
    "flask==3.1.0", "joblib==1.6.0", "jedi"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "check"])

print("Python:", sys.version.split()[0])
print({p.split("==")[0]: metadata.version(p.split("==")[0]) for p in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0', 'jedi': '0.20.1'}


In [16]:
import subprocess
import sys
import importlib.metadata as metadata
packages=['qiskit==2.2.3','numpy==2.2.6','scipy==1.15.3',
          'scikit-learn==1.6.1','pandas==2.2.3','matplotlib==3.10.1',
          'flask==3.1.0','joblib==1.6.0']
subprocess.check_call([sys.executable,'-m','pip','install',*packages])
subprocess.check_call([sys.executable,'-m','pip','check'])
print('Python:',sys.version.split()[0])
print({item.split('==')[0]:metadata.version(item.split('==')[0]) for item in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0'}


In [18]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
noise,response=7.,0.4
angles=np.array([noise/10,response/1.2])*np.pi # 세타: 회전각, 파이: 원주율
qc=QuantumCircuit(2)
qc.ry(float(angles[0]),0);qc.ry(float(angles[1]),1);qc.cx(0,1)
print('angles rad:',np.round(angles,6))
print('probabilities [00,01,10,11]:',np.round(Statevector.from_instruction(qc).probabilities(),6))

angles rad: [2.199115 1.047198]
probabilities [00,01,10,11]: [0.154581 0.198473 0.051527 0.595419]


In [19]:
import numpy as np
import threading
from flask import Flask,request,jsonify,render_template_string
from werkzeug.serving import make_server
from urllib.request import urlopen
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix,accuracy_score
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
# 열: 소음지수 0~10, 제동 응답시간 초, 우선점검 분류 0/1
# 하드코딩한 교육용 관측값과 규칙: 소음>=7 또는 응답시간>=0.8이면 1
rows=np.array([[1,.2,0],[2,.3,0],[3,.4,0],[4,.5,0],[5,.6,0],[6,.7,0],
 [1,.5,0],[4,.2,0],[7,.2,1],[8,.3,1],[9,.4,1],[10,.5,1],
 [2,.8,1],[3,.9,1],[4,1.,1],[8,.9,1],
 [2,.4,0],[5,.5,0],[7,.4,1],[5,.9,1]],float)
X=rows[:,:2]; y=rows[:,2].astype(int)
assert np.array_equal(y,((X[:,0]>=7)|(X[:,1]>=.8)).astype(int))
train=np.arange(16); test=np.arange(16,20)
def encode(values):
    vectors=[]
    for noise,response in values:
        angles=np.array([noise/10,response/1.2])*np.pi
        qc=QuantumCircuit(2)
        qc.ry(float(angles[0]),0); qc.ry(float(angles[1]),1)
        qc.cx(0,1)
        vectors.append(Statevector.from_instruction(qc).data)
    return np.array(vectors)
def kernel(a,b):
    return np.clip(abs(a.conj()@b.T)**2,0,1)
sa=encode(X[train]); k=kernel(sa,sa)
assert np.allclose(k,k.T) and np.allclose(k.diagonal(),1)
model=SVC(kernel='precomputed',C=10).fit(k,y[train])
pred=model.predict(kernel(encode(X[test]),sa))
print('accuracy:',accuracy_score(y[test],pred))
print('confusion [0,1]:',confusion_matrix(y[test],pred,labels=[0,1]).tolist())
app=Flask(__name__)
page='<!doctype html><html lang="ko"><meta charset="utf-8"><title>차량 우선점검 분류</title>\n<main style="max-width:650px;margin:40px auto;font-family:sans-serif"><h1>차량 우선점검 분류</h1><p>교육용 고정 규칙을 학습한 결과입니다. 실제 운행 판단에는 정비 점검이 필요합니다.</p><form id="form"><p><label>소음지수 0~10 <input name="noise" type="number" min="0" max="10" step="0.1" value="7" required></label></p><p><label>제동 응답시간 0~1.2초 <input name="response" type="number" min="0" max="1.2" step="0.01" value="0.4" required></label></p><button>분류 실행</button></form><pre id="result"></pre><script>document.getElementById(\'form\').onsubmit=async event=>{event.preventDefault();const form=new FormData(event.target);try{const r=await fetch(\'/predict\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({noise:Number(form.get(\'noise\')),response:Number(form.get(\'response\'))})});const d=await r.json();document.getElementById(\'result\').textContent=JSON.stringify(d,null,2);}catch(e){document.getElementById(\'result\').textContent=e.message;}};</script></main></html>'
@app.get('/')
def index(): return render_template_string(page)
@app.post('/predict')
def predict():
    body=request.get_json(silent=True)
    try:
        if not isinstance(body,dict): raise ValueError()
        values=[body['noise'],body['response']]
        if any(type(v) not in (int,float) for v in values): raise ValueError()
        row=np.array(values,float)
        if not np.isfinite(row).all() or not(0<=row[0]<=10 and 0<=row[1]<=1.2): raise ValueError()
    except (KeyError,ValueError,TypeError):
        return jsonify(error='소음 0~10, 응답시간 0~1.2초를 숫자로 입력해 주세요.'),400
    ktest=kernel(encode(row.reshape(1,-1)),sa)
    prediction=int(model.predict(ktest)[0])
    rule=int(row[0]>=7 or row[1]>=.8)
    return jsonify(prediction=prediction,rule=rule,
        meaning='우선점검' if prediction==1 else '일반점검',
        margin=float(model.decision_function(ktest)[0]))
client=app.test_client()
assert client.get('/').status_code==200
assert client.post('/predict',json={'noise':7,'response':.4}).status_code==200
for bad in [{},{'noise':11,'response':.4},{'noise':True,'response':.4},{'noise':2,'response':-1}]:
    assert client.post('/predict',json=bad).status_code==400
print('web tests: normal 200; invalid 400')
PORT=5003
if 'server' in globals(): server.shutdown()
server=make_server('0.0.0.0',PORT,app)
threading.Thread(target=server.serve_forever,daemon=True).start()
assert urlopen(f'http://127.0.0.1:{PORT}').status==200
print('internal HTTP: 200')

# Google Colab에서 공용 주소로 접속합니다.
import platform, subprocess, re, time
from pathlib import Path
from urllib.request import urlretrieve
binary=Path('/tmp/cloudflared_day34')
architecture='arm64' if platform.machine()=='aarch64' else 'amd64'
if not binary.exists():
    urlretrieve(f'https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-{architecture}',binary)
    binary.chmod(0o755)
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
log_path=Path(f'tunnel_{PORT}.log')
log=log_path.open('w')
tunnel=subprocess.Popen([str(binary),'tunnel','--url',f'http://127.0.0.1:{PORT}',
    '--no-autoupdate','--protocol','http2'],stdout=log,stderr=log)
public_url=None
for _ in range(45):
    time.sleep(1)
    match=re.search(r'https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com',log_path.read_text())
    if match:
        public_url=match.group(0); break
    if tunnel.poll() is not None:
        break
if public_url:
    print('브라우저 접속 URL:',public_url)
else:
    print('공용 URL 생성 실패:',str(log_path),'연결 상태를 확인합니다.')
    tunnel.terminate()
# Colab 커널이 실행되는 동안 서버와 터널이 유지됩니다.

accuracy: 1.0
confusion [0,1]: [[2, 0], [0, 2]]
web tests: normal 200; invalid 400


INFO:werkzeug:127.0.0.1 - - [11/Nov/2026 13:07:29] "GET / HTTP/1.1" 200 -


internal HTTP: 200
브라우저 접속 URL: https://issues-workout-fruits-hobbies.trycloudflare.com


고정 평가 표본 네 개에서 정확도는 1.0이고 혼동행렬은 [[2,0],[0,2]]입니다. 이 결과는 네 교육용 표본에서의 규칙 재현 결과입니다. 규칙 경계 주변의 모든 입력이나 실제 차량의 점검 성능을 나타내지는 않습니다.

웹의 `prediction`은 학습 모델 예측, `rule`은 입력에 직접 적용한 규칙 결과입니다. 두 값이 다르면 해당 입력에서 모델이 규칙을 재현하지 못한 것입니다. `meaning`은 우선점검 또는 일반점검이며, 실제 운행 가능 여부의 판단에는 별도의 정비 점검이 필요합니다. 서버는 범위를 벗어난 값과 잘못된 자료형을 HTTP 400으로 반환합니다. 입력과 출력 화면을 스크린샷으로 기록합니다.